# build_vocab: the PolyPlex v2 text vocabulary

This notebook decides the final text vocabulary (SPEC S2.3, S12.1a, S12.4, S12.6). Running it top to bottom produces the
same vocabulary again. Every step records in a table what it added, by which rule and from which data.

**Order of inclusion (the three tiers of S2.3)**
1. **Character floor**: every standard character of each language becomes a single token, with no measurement. The check is
   0% byte-fallback tokens on spoken data.
2. **Frequent word forms**: short, frequent spoken forms. The amount is set by the word cut rate (S2.2).
3. **Subwords**: pieces that cut the words tier 2 lacks into fewer tokens; sized by the cut rate where tier 2 does not
   reach the target, small elsewhere.

**Blocks (S12.1a, S12.4)**
- Base block: all 32,000 Helium tokens, plus Qwen3's digits, punctuation, symbols and byte fragments. Always computed.
- Script blocks: Qwen3 tokens with no identical string in Helium, grouped by script.
- Language blocks: the tokens the three tiers add.

**Output**: one result file per step under `.cache/huggingface/runs/vocab/`.

The code stays in this notebook (AGENTS.md 6.1). Run it from the repository root.

In [1]:
import collections
import json
import os
import re
import unicodedata
from dataclasses import dataclass
import urllib.request
from pathlib import Path

os.environ.setdefault("HF_HOME", str(Path(".cache/huggingface").resolve()))
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from IPython.display import Markdown, display
from transformers import AutoTokenizer

from polyplex_research.utils.assemble import load_base_tokenizer

HELIUM = "nvidia/personaplex-7b-v1"
QWEN = "Qwen/Qwen3-8B"
# CLDR release the standard characters are read from. A fixed tag keeps the result reproducible.
CLDR_RELEASE = "48.2.3"
OUT = Path(".cache/huggingface/runs/vocab")


@dataclass(frozen=True)
class Qwen3Language:
    """One language or dialect Qwen3 lists, with its code in each corpus that has it (None when the corpus lacks it)."""

    name: str
    fineweb2: str | None
    wikipedia: str | None
    fleurs: str | None


# The 119 languages and dialects of the Qwen3 release (https://qwenlm.github.io/blog/qwen3/), in its order and grouped by
# its language families. Codes checked against the Hugging Face listings of FineWeb-2, Wikipedia (20231101) and FLEURS.
# A variety without a corpus of its own shares its parent's code, as Dari shares Persian's, or has none.
QWEN3_LANGUAGES = (
    # Indo-European
    Qwen3Language("English", None, "en", "en_us"),
    Qwen3Language("French", "fra_Latn", "fr", "fr_fr"),
    Qwen3Language("Portuguese", "por_Latn", "pt", "pt_br"),
    Qwen3Language("German", "deu_Latn", "de", "de_de"),
    Qwen3Language("Romanian", "ron_Latn", "ro", "ro_ro"),
    Qwen3Language("Swedish", "swe_Latn", "sv", "sv_se"),
    Qwen3Language("Danish", "dan_Latn", "da", "da_dk"),
    Qwen3Language("Bulgarian", "bul_Cyrl", "bg", "bg_bg"),
    Qwen3Language("Russian", "rus_Cyrl", "ru", "ru_ru"),
    Qwen3Language("Czech", "ces_Latn", "cs", "cs_cz"),
    Qwen3Language("Greek", "ell_Grek", "el", "el_gr"),
    Qwen3Language("Ukrainian", "ukr_Cyrl", "uk", "uk_ua"),
    Qwen3Language("Spanish", "spa_Latn", "es", "es_419"),
    Qwen3Language("Dutch", "nld_Latn", "nl", "nl_nl"),
    Qwen3Language("Slovak", "slk_Latn", "sk", "sk_sk"),
    Qwen3Language("Croatian", "hrv_Latn", "hr", "hr_hr"),
    Qwen3Language("Polish", "pol_Latn", "pl", "pl_pl"),
    Qwen3Language("Lithuanian", "lit_Latn", "lt", "lt_lt"),
    Qwen3Language("Norwegian Bokmal", "nob_Latn", "no", "nb_no"),
    Qwen3Language("Norwegian Nynorsk", "nno_Latn", "nn", None),
    Qwen3Language("Persian", "fas_Arab", "fa", "fa_ir"),
    Qwen3Language("Slovenian", "slv_Latn", "sl", "sl_si"),
    Qwen3Language("Gujarati", "guj_Gujr", "gu", "gu_in"),
    Qwen3Language("Latvian", "lvs_Latn", "lv", "lv_lv"),
    Qwen3Language("Italian", "ita_Latn", "it", "it_it"),
    Qwen3Language("Occitan", "oci_Latn", "oc", "oc_fr"),
    Qwen3Language("Nepali", "npi_Deva", "ne", "ne_np"),
    Qwen3Language("Marathi", "mar_Deva", "mr", "mr_in"),
    Qwen3Language("Belarusian", "bel_Cyrl", "be", "be_by"),
    Qwen3Language("Serbian", "srp_Cyrl", "sr", "sr_rs"),
    Qwen3Language("Luxembourgish", "ltz_Latn", "lb", "lb_lu"),
    Qwen3Language("Venetian", "vec_Latn", "vec", None),
    Qwen3Language("Assamese", "asm_Beng", "as", "as_in"),
    Qwen3Language("Welsh", "cym_Latn", "cy", "cy_gb"),
    Qwen3Language("Silesian", "szl_Latn", "szl", None),
    Qwen3Language("Asturian", "ast_Latn", "ast", "ast_es"),
    Qwen3Language("Chhattisgarhi", "hne_Deva", None, None),
    Qwen3Language("Awadhi", "awa_Deva", None, None),
    Qwen3Language("Maithili", "mai_Deva", "mai", None),
    Qwen3Language("Bhojpuri", "bho_Deva", "bh", None),
    Qwen3Language("Sindhi", "snd_Arab", "sd", "sd_in"),
    Qwen3Language("Irish", "gle_Latn", "ga", "ga_ie"),
    Qwen3Language("Faroese", "fao_Latn", "fo", None),
    Qwen3Language("Hindi", "hin_Deva", "hi", "hi_in"),
    Qwen3Language("Punjabi", "pan_Guru", "pa", "pa_in"),
    Qwen3Language("Bengali", "ben_Beng", "bn", "bn_in"),
    Qwen3Language("Oriya", "ory_Orya", "or", "or_in"),
    Qwen3Language("Tajik", "tgk_Cyrl", "tg", "tg_tj"),
    Qwen3Language("Eastern Yiddish", "ydd_Hebr", "yi", None),
    Qwen3Language("Lombard", "lmo_Latn", "lmo", None),
    Qwen3Language("Ligurian", "lij_Latn", "lij", None),
    Qwen3Language("Sicilian", "scn_Latn", "scn", None),
    Qwen3Language("Friulian", "fur_Latn", "fur", None),
    Qwen3Language("Sardinian", "srd_Latn", "sc", None),
    Qwen3Language("Galician", "glg_Latn", "gl", "gl_es"),
    Qwen3Language("Catalan", "cat_Latn", "ca", "ca_es"),
    Qwen3Language("Icelandic", "isl_Latn", "is", "is_is"),
    Qwen3Language("Tosk Albanian", "als_Latn", "sq", None),
    Qwen3Language("Limburgish", "lim_Latn", "li", None),
    Qwen3Language("Dari", "fas_Arab", None, None),
    Qwen3Language("Afrikaans", "afr_Latn", "af", "af_za"),
    Qwen3Language("Macedonian", "mkd_Cyrl", "mk", "mk_mk"),
    Qwen3Language("Sinhala", "sin_Sinh", "si", None),
    Qwen3Language("Urdu", "urd_Arab", "ur", "ur_pk"),
    Qwen3Language("Magahi", "mag_Deva", None, None),
    Qwen3Language("Bosnian", "bos_Latn", "bs", "bs_ba"),
    Qwen3Language("Armenian", "hye_Armn", "hy", "hy_am"),
    # Sino-Tibetan
    Qwen3Language("Simplified Chinese", "cmn_Hani", "zh", "cmn_hans_cn"),
    Qwen3Language("Traditional Chinese", "cmn_Hani", "zh", None),
    Qwen3Language("Cantonese", "yue_Hani", "zh-yue", "yue_hant_hk"),
    Qwen3Language("Burmese", "mya_Mymr", "my", "my_mm"),
    # Afro-Asiatic
    Qwen3Language("Standard Arabic", "arb_Arab", "ar", "ar_eg"),
    Qwen3Language("Najdi Arabic", "ars_Arab", None, None),
    Qwen3Language("Levantine Arabic", "apc_Arab", None, None),
    Qwen3Language("Egyptian Arabic", "arz_Arab", "arz", None),
    Qwen3Language("Moroccan Arabic", "ary_Arab", "ary", None),
    Qwen3Language("Mesopotamian Arabic", "acm_Arab", None, None),
    Qwen3Language("Taizzi-Adeni Arabic", None, None, None),
    Qwen3Language("Tunisian Arabic", "aeb_Arab", None, None),
    Qwen3Language("Hebrew", "heb_Hebr", "he", "he_il"),
    Qwen3Language("Maltese", "mlt_Latn", "mt", "mt_mt"),
    # Austronesian
    Qwen3Language("Indonesian", "ind_Latn", "id", "id_id"),
    Qwen3Language("Malay", "zsm_Latn", "ms", "ms_my"),
    Qwen3Language("Tagalog", "fil_Latn", "tl", "fil_ph"),
    Qwen3Language("Cebuano", "ceb_Latn", "ceb", "ceb_ph"),
    Qwen3Language("Javanese", "jav_Latn", "jv", "jv_id"),
    Qwen3Language("Sundanese", "sun_Latn", "su", None),
    Qwen3Language("Minangkabau", "min_Latn", "min", None),
    Qwen3Language("Balinese", "ban_Latn", "ban", None),
    Qwen3Language("Banjar", "bjn_Latn", "bjn", None),
    Qwen3Language("Pangasinan", "pag_Latn", "pag", None),
    Qwen3Language("Iloko", "ilo_Latn", "ilo", None),
    Qwen3Language("Waray", "war_Latn", "war", None),
    # Dravidian
    Qwen3Language("Tamil", "tam_Taml", "ta", "ta_in"),
    Qwen3Language("Telugu", "tel_Telu", "te", "te_in"),
    Qwen3Language("Kannada", "kan_Knda", "kn", "kn_in"),
    Qwen3Language("Malayalam", "mal_Mlym", "ml", "ml_in"),
    # Turkic
    Qwen3Language("Turkish", "tur_Latn", "tr", "tr_tr"),
    Qwen3Language("North Azerbaijani", "azj_Latn", "az", "az_az"),
    Qwen3Language("Northern Uzbek", "uzn_Latn", "uz", "uz_uz"),
    Qwen3Language("Kazakh", "kaz_Cyrl", "kk", "kk_kz"),
    Qwen3Language("Bashkir", "bak_Cyrl", "ba", None),
    Qwen3Language("Tatar", "tat_Cyrl", "tt", None),
    # Tai-Kadai
    Qwen3Language("Thai", "tha_Thai", "th", "th_th"),
    Qwen3Language("Lao", "lao_Laoo", "lo", "lo_la"),
    # Uralic
    Qwen3Language("Finnish", "fin_Latn", "fi", "fi_fi"),
    Qwen3Language("Estonian", "ekk_Latn", "et", "et_ee"),
    Qwen3Language("Hungarian", "hun_Latn", "hu", "hu_hu"),
    # Austroasiatic
    Qwen3Language("Vietnamese", "vie_Latn", "vi", "vi_vn"),
    Qwen3Language("Khmer", "khm_Khmr", "km", "km_kh"),
    # Other
    Qwen3Language("Japanese", "jpn_Jpan", "ja", "ja_jp"),
    Qwen3Language("Korean", "kor_Hang", "ko", "ko_kr"),
    Qwen3Language("Georgian", "kat_Geor", "ka", "ka_ge"),
    Qwen3Language("Basque", "eus_Latn", "eu", None),
    Qwen3Language("Haitian", "hat_Latn", "ht", None),
    Qwen3Language("Papiamento", "pap_Latn", "pap", None),
    Qwen3Language("Kabuverdianu", "kea_Latn", None, "kea_cv"),
    Qwen3Language("Tok Pisin", "tpi_Latn", "tpi", None),
    Qwen3Language("Swahili", "swh_Latn", "sw", "sw_ke"),
)

OUT.mkdir(parents=True, exist_ok=True)


import time

THREADS = os.cpu_count()
started = time.time()
# Languages are searched independently, each in a forked process with a share of the cores.
WORKERS = max(1, min(32, THREADS // 2))
WORKER_THREADS = max(1, THREADS // WORKERS)


def in_workers(function, items):
    if WORKERS == 1:
        return [function(item) for item in items]
    import multiprocessing

    with multiprocessing.get_context("fork").Pool(WORKERS) as pool:
        return pool.map(function, items, chunksize=1)


def elapsed(label):
    # Also appended to a log file, so a running notebook shows its progress.
    line = f"[{label}] {time.time() - started:,.0f} s since the start"
    print(line, flush=True)
    with open(OUT / "build_vocab_progress.log", "a") as log:
        log.write(f"{time.strftime('%H:%M:%S')} {line}\n")


def table(rows, header):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

W1024 18:59:12.231000 3827323 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1024 18:59:12.301000 3827323 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


## 1. Matching the base vocabularies

Qwen3 and Helium tokens are matched by surface string. Helium's `▁` (SentencePiece) and Qwen3's `Ġ` (byte BPE) both mean a
leading space. Tokens with the same string are treated as one token and kept as the Helium token (base block, Helium head rows
preserved). Qwen3 tokens missing from Helium are grouped by script into script blocks. Digits, punctuation, symbols and byte
fragments go to the base block.

In [2]:
helium = load_base_tokenizer(HELIUM)
qwen = AutoTokenizer.from_pretrained(QWEN)

helium_surface = {}
helium_special = []
for index in range(len(helium)):
    piece = helium.convert_ids_to_tokens(index)
    if piece.startswith("<") and piece.endswith(">"):
        helium_special.append(piece)
        continue
    helium_surface.setdefault(piece.replace("▁", " "), index)

# Qwen3's added tokens are markers, not text. Its tool-call and thinking markers enter as control tokens; the fill-in-the-middle
# and repository markers are left out (S12.1a). A Qwen3 string equal to a Helium control token (`<s>`) is left out too.
QWEN_CONTROL = ["<tool_call>", "</tool_call>", "<tool_response>", "</tool_response>", "<think>", "</think>"]
qwen_surface = {}
for token, index in qwen.get_vocab().items():
    if index in qwen.all_special_ids or token in qwen.get_added_vocab():
        continue
    surface = qwen.convert_tokens_to_string([token])
    if surface not in helium_special:
        qwen_surface.setdefault(surface, index)

shared = set(helium_surface) & set(qwen_surface)
helium_only = set(helium_surface) - set(qwen_surface)

SCRIPTS = ("LATIN", "CYRILLIC", "ARABIC", "HEBREW", "GREEK", "THAI", "LAO", "KHMER", "MYANMAR", "GEORGIAN", "ARMENIAN",
           "DEVANAGARI", "BENGALI", "GURMUKHI", "GUJARATI", "ORIYA", "TAMIL", "TELUGU", "KANNADA", "MALAYALAM", "SINHALA")


def script_of(ch):
    code = ord(ch)
    if 0xAC00 <= code <= 0xD7A3 or 0x1100 <= code <= 0x11FF or 0x3130 <= code <= 0x318F:
        return "hangul"
    if 0x3040 <= code <= 0x30FF or 0x31F0 <= code <= 0x31FF:
        return "kana"
    if 0x4E00 <= code <= 0x9FFF or 0x3400 <= code <= 0x4DBF or 0xF900 <= code <= 0xFAFF or 0x20000 <= code <= 0x2FFFF:
        return "han"
    if ch.isascii():
        return "latin" if ch.isalpha() else None
    name = unicodedata.name(ch, "")
    for script in SCRIPTS:
        if name.startswith(script):
            return script.lower()
    if unicodedata.category(ch)[0] in "PSZNC":
        return None
    return "other"


def block_of(surface):
    if "�" in surface or not surface.strip():
        return "base"
    scripts = {script_of(ch) for ch in surface} - {None}
    if not scripts:
        return "base"
    if scripts <= {"han", "kana"}:
        return "kana" if "kana" in scripts else "han"
    if len(scripts) == 1:
        return next(iter(scripts))
    return "mixed"


qwen_blocks = collections.defaultdict(list)
for surface, index in qwen_surface.items():
    if surface in shared:
        continue
    qwen_blocks[block_of(surface)].append(surface)

base_size = len(helium) + len(QWEN_CONTROL) + len(qwen_blocks["base"])
table(
    [
        ["Helium vocabulary (with special tokens)", len(helium)],
        ["  of which special tokens", len(helium_special)],
        ["Qwen3 vocabulary (without special and added tokens)", len(qwen_surface)],
        ["Qwen3 tool-call and thinking markers (control tokens)", len(QWEN_CONTROL)],
        ["Tokens with the same string in both (kept as Helium tokens)", len(shared)],
        ["Helium-only tokens", len(helium_only)],
        ["Qwen3-only digits, punctuation, symbols and byte fragments (base block)", len(qwen_blocks["base"])],
        ["**Base block total**", base_size],
    ],
    ["Item", "Count"],
)
table(
    sorted(([name, len(tokens)] for name, tokens in qwen_blocks.items() if name != "base"), key=lambda row: -row[1]),
    ["Script block (Qwen3-only tokens)", "Count"],
)
json.dump({name: sorted(tokens) for name, tokens in qwen_blocks.items()}, open(OUT / "qwen_blocks.json", "w"), ensure_ascii=False)

| Item | Count |
|---|---|
| Helium vocabulary (with special tokens) | 32000 |
|   of which special tokens | 261 |
| Qwen3 vocabulary (without special and added tokens) | 150303 |
| Qwen3 tool-call and thinking markers (control tokens) | 6 |
| Tokens with the same string in both (kept as Helium tokens) | 26471 |
| Helium-only tokens | 5268 |
| Qwen3-only digits, punctuation, symbols and byte fragments (base block) | 8297 |
| **Base block total** | 40303 |

| Script block (Qwen3-only tokens) | Count |
|---|---|
| latin | 68839 |
| han | 25462 |
| arabic | 3992 |
| cyrillic | 3949 |
| hangul | 3564 |
| hebrew | 3164 |
| thai | 2570 |
| kana | 2067 |
| other | 1231 |
| greek | 193 |
| armenian | 77 |
| devanagari | 63 |
| bengali | 44 |
| myanmar | 37 |
| georgian | 37 |
| khmer | 36 |
| lao | 33 |
| malayalam | 32 |
| tamil | 28 |
| sinhala | 26 |
| kannada | 25 |
| mixed | 19 |
| gujarati | 16 |
| telugu | 14 |
| gurmukhi | 10 |
| oriya | 7 |

## 2. Tier 1: the character floor

Each language gets every character of its standard lists, with no measurement (S2.3). Tier 1 comes from the standard lists
only and is never extended from data.

| Applies to | Standard list |
|---|---|
| All languages | CLDR exemplar characters (`exemplarCharacters`), punctuation (`punctuation`), and the digits of the default and native (`native`) numbering systems. Upper case too, where the script has case |
| Indic, Southeast Asian and Georgian scripts | In addition, every assigned character of the script's Unicode block, plus ZWJ and ZWNJ |
| Korean | Hangul from the 2,350 KS X 1001 syllables |
| Japanese | Kanji from JIS X 0208 levels 1 and 2 |
| Simplified Chinese | Hanzi from GB 2312 |
| Traditional Chinese, Cantonese | Hanzi from the 5,401 Big5 common characters (常用字) |

A language missing from CLDR takes its parent language's set, noted in the table. Characters that are already single Qwen3
tokens are not added again. Non-standard characters found in spoken data are kept aside as candidates for tiers 2 and 3.

In [3]:
CLDR_LOCALE = {
    "English": "en", "Norwegian Bokmal": "nb", "Norwegian Nynorsk": "nn", "Persian": "fa", "Dari": "fa-AF",
    "Simplified Chinese": "zh", "Traditional Chinese": "zh-Hant", "Cantonese": "yue",
    "Standard Arabic": "ar", "Najdi Arabic": "ar-SA", "Levantine Arabic": "apc", "Egyptian Arabic": "ar-EG",
    "Moroccan Arabic": "ar-MA", "Mesopotamian Arabic": "ar-IQ", "Taizzi-Adeni Arabic": "ar-YE", "Tunisian Arabic": "ar-TN",
    "Tagalog": "fil", "Malay": "ms", "Serbian": "sr", "Tosk Albanian": "sq", "Eastern Yiddish": "yi",
    "Northern Uzbek": "uz", "North Azerbaijani": "az", "Latvian": "lv", "Estonian": "et", "Nepali": "ne", "Swahili": "sw",
    "Bhojpuri": "bho", "Maithili": "mai", "Kabuverdianu": "kea", "Sardinian": "sc",
}
# Languages CLDR does not describe, read from the closest language it does.
CLDR_PARENT = {
    "Chhattisgarhi": "hi", "Awadhi": "hi", "Magahi": "hi", "Venetian": "it", "Lombard": "it", "Ligurian": "it",
    "Sicilian": "it", "Friulian": "it", "Silesian": "pl", "Limburgish": "nl", "Minangkabau": "id", "Balinese": "id",
    "Banjar": "id", "Pangasinan": "fil", "Iloko": "fil", "Waray": "fil", "Papiamento": "es", "Tok Pisin": "en",
    "Haitian": "fr", "Tatar": "ru", "Bashkir": "ru",
}

cache = Path(".cache/cldr") / CLDR_RELEASE
cache.mkdir(parents=True, exist_ok=True)


def cldr_file(package, locale, name):
    path = cache / package / f"{locale}.{name}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        url = (f"https://raw.githubusercontent.com/unicode-org/cldr-json/{CLDR_RELEASE}/cldr-json/{package}/"
               + (f"main/{locale}/{name}.json" if locale else f"{name}.json"))
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                path.write_bytes(response.read())
        except urllib.error.HTTPError:
            return None
    return json.load(open(path))


def cldr_characters(locale):
    data = cldr_file("cldr-misc-full", locale, "characters")
    return None if data is None else data["main"][locale]["characters"]


NUMBERING_SYSTEMS = cldr_file("cldr-core", None, "supplemental/numberingSystems")["supplemental"]["numberingSystems"]


def cldr_digits(locale):
    data = cldr_file("cldr-numbers-full", locale, "numbers")
    if data is None:
        return set()
    numbers = data["main"][locale]["numbers"]
    systems = {numbers["defaultNumberingSystem"], *numbers.get("otherNumberingSystems", {}).values()}
    return {digit for system in systems for digit in NUMBERING_SYSTEMS.get(system, {}).get("_digits", "")}


def expand_exemplars(text):
    # CLDR writes combining marks and some symbols as \uXXXX escapes, and quotes other characters with a backslash.
    text = re.sub(r"\\u([0-9A-Fa-f]{4})", lambda match: chr(int(match.group(1), 16)), text).replace("\\", "")
    items = []
    for part in text.strip("[]").split():
        if part.startswith("{") and part.endswith("}"):
            items.append(part[1:-1])
        elif len(part) == 3 and part[1] == "-":
            items += [chr(code) for code in range(ord(part[0]), ord(part[2]) + 1)]
        else:
            items += list(part)
    with_case = set(items)
    with_case |= {item.upper() for item in items if item.upper() != item}
    return with_case


standard = {}
source = {}
for language in QWEN3_LANGUAGES:
    if language.name == "Korean":
        continue
    candidates = [CLDR_LOCALE.get(language.name), language.wikipedia, CLDR_PARENT.get(language.name)]
    for locale in [code for code in candidates if code]:
        characters = cldr_characters(locale)
        if characters is not None:
            standard[language.name] = (expand_exemplars(characters["exemplarCharacters"])
                                       | expand_exemplars(characters.get("punctuation", ""))
                                       | cldr_digits(locale))
            source[language.name] = locale if locale != CLDR_PARENT.get(language.name) else f"{locale} (related language)"
            break
missing_cldr = [language.name for language in QWEN3_LANGUAGES if language.name not in standard and language.name != "Korean"]
print("Languages with no CLDR exemplar characters:", missing_cldr)

Languages with no CLDR exemplar characters: []


### Korean: KS X 1001

The Korean spoken data is the v1 data: every utterance of the AI Hub daily conversation multi-session corpus (011) and the
Korean source text of the AI Hub colloquial Korean-English corpus.

In [4]:
KOREAN_SPOKEN = Path(".cache/datasets")


def korean_spoken_texts():
    for path in sorted((KOREAN_SPOKEN / "multisession_daily").rglob("*.json")):
        for session in json.load(open(path))["sessionInfo"]:
            for turn in session["dialog"]:
                yield turn["utterance"]
    for path in sorted((KOREAN_SPOKEN / "colloquial_koen").rglob("*한영*.json")):
        data = json.load(open(path))
        for row in data["data"] if isinstance(data, dict) else data:
            yield row["ko_original"]


korean_texts = list(korean_spoken_texts())
ks_x_1001 = {chr(code) for code in range(0xAC00, 0xD7A4) if len(chr(code).encode("euc-kr", "ignore")) == 2}
syllables_seen = collections.Counter(ch for text in korean_texts for ch in text if 0xAC00 <= ord(ch) <= 0xD7A3)
outside = {ch: count for ch, count in syllables_seen.items() if ch not in ks_x_1001}
korean_cldr = cldr_characters("ko")
standard["Korean"] = ks_x_1001 | expand_exemplars(korean_cldr["punctuation"]) | cldr_digits("ko")
# Syllables the spoken data uses outside KS X 1001 are candidates for tiers 2 and 3, not tier 1.
json.dump(outside, open(OUT / "korean_candidates_outside_ks.json", "w"), ensure_ascii=False, indent=1)
source["Korean"] = "KS X 1001 + CLDR punctuation and digits"
print(f"{len(korean_texts):,} spoken sentences, {len(syllables_seen):,} distinct syllables, {len(outside)} of them outside KS X 1001")
print("Syllables outside KS X 1001 kept as tier 2 and 3 candidates, most frequent first:", sorted(outside.items(), key=lambda kv: -kv[1])[:30])

3,273,481 spoken sentences, 2,501 distinct syllables, 424 of them outside KS X 1001
Syllables outside KS X 1001 kept as tier 2 and 3 candidates, most frequent first: [('됬', 1500), ('몐', 390), ('떄', 185), ('슌', 142), ('귬', 111), ('녓', 61), ('춉', 55), ('웻', 52), ('뱡', 46), ('먄', 43), ('곘', 41), ('똠', 40), ('븡', 37), ('렜', 36), ('픙', 36), ('녤', 34), ('댠', 33), ('됭', 33), ('퀜', 30), ('똣', 26), ('헀', 24), ('닏', 22), ('솁', 22), ('쎼', 21), ('늗', 21), ('럅', 20), ('뢱', 20), ('잌', 19), ('탼', 19), ('졋', 18)]


### Tier 1 tokens per language

A language's tier 1 tokens are its standard characters that are not single tokens in the Qwen3 vocabulary, and the word-start
form (`▁` plus the character, S12.1b) of every standard letter and digit that has no space-led form in Qwen3 or Helium.
Without that form a word starting with the character spends a frame on `▁` alone. Word-start forms are written here with a
leading space, as the other surface strings are.

In [5]:
# Scripts whose CLDR lists leave out vowel signs, letters with a dot or old letters, so the whole Unicode block is taken.
BLOCK_SCRIPTS = {
    "devanagari": (0x0900, 0x097F), "bengali": (0x0980, 0x09FF), "gurmukhi": (0x0A00, 0x0A7F), "gujarati": (0x0A80, 0x0AFF),
    "oriya": (0x0B00, 0x0B7F), "tamil": (0x0B80, 0x0BFF), "telugu": (0x0C00, 0x0C7F), "kannada": (0x0C80, 0x0CFF),
    "malayalam": (0x0D00, 0x0D7F), "sinhala": (0x0D80, 0x0DFF), "thai": (0x0E00, 0x0E7F), "lao": (0x0E80, 0x0EFF),
    "myanmar": (0x1000, 0x109F), "georgian": (0x10A0, 0x10FF), "khmer": (0x1780, 0x17FF),
}


# Zero width joiner and non-joiner, which these scripts' orthographies use between letters.
JOINERS = {"\u200c", "\u200d"}


def block_letters(chars):
    letters = set()
    for script in {script_of(ch) for ch in chars if len(ch) == 1 and ch.isalpha()} & set(BLOCK_SCRIPTS):
        start, end = BLOCK_SCRIPTS[script]
        letters |= {chr(code) for code in range(start, end + 1) if unicodedata.name(chr(code), "")}
        letters |= JOINERS
    return letters


def encoding_hanzi(codec, keep=lambda encoded: True):
    hanzi = set()
    for code in list(range(0x4E00, 0xA000)) + list(range(0x3400, 0x4DC0)) + list(range(0xF900, 0xFB00)):
        try:
            encoded = chr(code).encode(codec)
        except UnicodeEncodeError:
            continue
        if keep(encoded):
            hanzi.add(chr(code))
    return hanzi


def big5_common(encoded):
    # Big5 level 1 (常用字) runs from A440 to C67E.
    return len(encoded) == 2 and (0xA4 <= encoded[0] < 0xC6 or (encoded[0] == 0xC6 and encoded[1] <= 0x7E))


HANZI_STANDARD = {
    "Japanese": ("JIS X 0208", encoding_hanzi("euc_jp", lambda encoded: len(encoded) == 2)),
    "Simplified Chinese": ("GB 2312", encoding_hanzi("gb2312")),
    "Traditional Chinese": ("Big5 common characters", encoding_hanzi("big5", big5_common)),
    "Cantonese": ("Big5 common characters", encoding_hanzi("big5", big5_common)),
}
for name, (label, hanzi) in HANZI_STANDARD.items():
    # The encoding standard replaces CLDR's ideographs; kana, punctuation and digits stay as CLDR lists them.
    standard[name] = {ch for ch in standard[name] if script_of(ch) != "han"} | hanzi
    source[name] = f"{source[name]}, Han characters from {label}"
for language in QWEN3_LANGUAGES:
    extra = block_letters(standard.get(language.name, set()))
    if extra - standard.get(language.name, set()):
        standard[language.name] = standard[language.name] | extra
        source[language.name] = f"{source[language.name]} + Unicode block"

spaced = set(qwen_surface) | set(helium_surface)
tier1 = {}
rows = []
for language in QWEN3_LANGUAGES:
    chars = standard.get(language.name, set())
    added = sorted(ch for ch in chars if ch not in qwen_surface)
    starts = sorted(" " + ch for ch in chars if len(ch) == 1 and (ch.isalpha() or ch.isdigit()) and " " + ch not in spaced)
    tier1[language.name] = added + starts
    rows.append([language.name, source.get(language.name, "none"), len(chars), len(chars) - len(added), len(added), len(starts)])
table(rows, ["Language", "Standard source", "Standard characters", "Already in Qwen3", "Added as tier 1", "Word-start forms added"])
all_tier1 = sorted({ch for chars in tier1.values() for ch in chars})
print(f"Tier 1 tokens summed over languages: {sum(len(v) for v in tier1.values()):,}, distinct: {len(all_tier1):,}")
json.dump({"cldr_release": CLDR_RELEASE, "source": source, "tier1": tier1}, open(OUT / "tier1.json", "w"), ensure_ascii=False, indent=1)

| Language | Standard source | Standard characters | Already in Qwen3 | Added as tier 1 | Word-start forms added |
|---|---|---|---|---|---|
| English | en | 94 | 94 | 0 | 10 |
| French | fr | 124 | 124 | 0 | 23 |
| Portuguese | pt | 120 | 120 | 0 | 19 |
| German | de | 102 | 102 | 0 | 11 |
| Romanian | ro | 99 | 99 | 0 | 16 |
| Swedish | sv | 104 | 104 | 0 | 10 |
| Danish | da | 98 | 98 | 0 | 11 |
| Bulgarian | bg | 98 | 98 | 0 | 19 |
| Russian | ru | 108 | 108 | 0 | 23 |
| Czech | cs | 118 | 118 | 0 | 28 |
| Greek | el | 104 | 101 | 3 | 50 |
| Ukrainian | uk | 104 | 104 | 0 | 25 |
| Spanish | es | 112 | 112 | 0 | 15 |
| Dutch | nl | 116 | 114 | 2 | 17 |
| Slovak | sk | 126 | 123 | 3 | 28 |
| Croatian | hr | 97 | 94 | 3 | 14 |
| Polish | pl | 111 | 111 | 0 | 21 |
| Lithuanian | lt | 94 | 93 | 1 | 24 |
| Norwegian Bokmal | nb | 110 | 110 | 0 | 14 |
| Norwegian Nynorsk | nn | 110 | 110 | 0 | 14 |
| Persian | fa | 75 | 62 | 13 | 19 |
| Slovenian | sl | 84 | 84 | 0 | 12 |
| Gujarati | gu + Unicode block | 135 | 59 | 76 | 73 |
| Latvian | lv | 110 | 110 | 0 | 28 |
| Italian | it | 102 | 102 | 0 | 16 |
| Occitan | oc | 115 | 115 | 0 | 17 |
| Nepali | ne + Unicode block | 149 | 81 | 68 | 86 |
| Marathi | mr + Unicode block | 161 | 91 | 70 | 86 |
| Belarusian | be | 94 | 91 | 3 | 22 |
| Serbian | sr | 93 | 93 | 0 | 22 |
| Luxembourgish | lb | 100 | 100 | 0 | 12 |
| Venetian | vec | 108 | 107 | 1 | 17 |
| Assamese | as + Unicode block | 134 | 80 | 54 | 65 |
| Welsh | cy | 156 | 153 | 3 | 41 |
| Silesian | szl | 115 | 115 | 0 | 23 |
| Asturian | ast | 110 | 109 | 1 | 19 |
| Chhattisgarhi | hi (related language) + Unicode block | 171 | 103 | 68 | 96 |
| Awadhi | hi (related language) + Unicode block | 171 | 103 | 68 | 96 |
| Maithili | mai + Unicode block | 180 | 108 | 72 | 96 |
| Bhojpuri | bho + Unicode block | 172 | 104 | 68 | 96 |
| Sindhi | sd | 82 | 64 | 18 | 33 |
| Irish | ga | 88 | 88 | 0 | 13 |
| Faroese | fo | 98 | 98 | 0 | 17 |
| Hindi | hi + Unicode block | 171 | 103 | 68 | 96 |
| Punjabi | pa + Unicode block | 119 | 48 | 71 | 71 |
| Bengali | bn + Unicode block | 134 | 80 | 54 | 65 |
| Oriya | or + Unicode block | 139 | 53 | 86 | 73 |
| Tajik | tg | 113 | 110 | 3 | 29 |
| Eastern Yiddish | yi | 74 | 61 | 13 | 16 |
| Lombard | lmo | 87 | 87 | 0 | 10 |
| Ligurian | lij | 132 | 132 | 0 | 27 |
| Sicilian | scn | 108 | 108 | 0 | 21 |
| Friulian | fur | 99 | 99 | 0 | 19 |
| Sardinian | sc | 92 | 92 | 0 | 16 |
| Galician | gl | 114 | 114 | 0 | 17 |
| Catalan | ca | 119 | 119 | 0 | 17 |
| Icelandic | is | 106 | 106 | 0 | 18 |
| Tosk Albanian | sq | 115 | 111 | 4 | 12 |
| Limburgish | nl (related language) | 116 | 114 | 2 | 17 |
| Dari | fa-AF | 75 | 62 | 13 | 19 |
| Afrikaans | af | 116 | 116 | 0 | 17 |
| Macedonian | mk | 95 | 95 | 0 | 24 |
| Sinhala | si + Unicode block | 135 | 69 | 66 | 79 |
| Urdu | ur | 70 | 60 | 10 | 32 |
| Magahi | hi (related language) + Unicode block | 171 | 103 | 68 | 96 |
| Bosnian | bs | 96 | 93 | 3 | 14 |
| Armenian | hy | 98 | 90 | 8 | 86 |
| Simplified Chinese | zh, Han characters from GB 2312 | 6828 | 6705 | 123 | 6727 |
| Traditional Chinese | zh-Hant, Han characters from Big5 common characters | 5543 | 4119 | 1424 | 5377 |
| Cantonese | yue, Han characters from Big5 common characters | 5543 | 4119 | 1424 | 5377 |
| Burmese | my + Unicode block | 182 | 58 | 124 | 94 |
| Standard Arabic | ar | 84 | 71 | 13 | 26 |
| Najdi Arabic | ar-SA | 74 | 61 | 13 | 16 |
| Levantine Arabic | apc | 76 | 73 | 3 | 16 |
| Egyptian Arabic | ar-EG | 74 | 61 | 13 | 16 |
| Moroccan Arabic | ar-MA | 84 | 71 | 13 | 26 |
| Mesopotamian Arabic | ar-IQ | 74 | 61 | 13 | 16 |
| Taizzi-Adeni Arabic | ar-YE | 74 | 61 | 13 | 16 |
| Tunisian Arabic | ar-TN | 84 | 71 | 13 | 26 |
| Hebrew | he | 58 | 58 | 0 | 15 |
| Maltese | mt | 98 | 96 | 2 | 22 |
| Indonesian | id | 94 | 94 | 0 | 10 |
| Malay | ms | 94 | 94 | 0 | 10 |
| Tagalog | fil | 95 | 95 | 0 | 12 |
| Cebuano | ceb | 74 | 74 | 0 | 10 |
| Javanese | jv | 93 | 83 | 10 | 27 |
| Sundanese | su | 96 | 96 | 0 | 10 |
| Minangkabau | id (related language) | 94 | 94 | 0 | 10 |
| Balinese | id (related language) | 94 | 94 | 0 | 10 |
| Banjar | id (related language) | 94 | 94 | 0 | 10 |
| Pangasinan | fil (related language) | 95 | 95 | 0 | 12 |
| Iloko | fil (related language) | 95 | 95 | 0 | 12 |
| Waray | fil (related language) | 95 | 95 | 0 | 12 |
| Tamil | ta + Unicode block | 116 | 71 | 45 | 57 |
| Telugu | te + Unicode block | 134 | 47 | 87 | 77 |
| Kannada | kn + Unicode block | 135 | 68 | 67 | 77 |
| Malayalam | ml + Unicode block | 165 | 77 | 88 | 87 |
| Turkish | tr | 100 | 100 | 0 | 13 |
| North Azerbaijani | az | 106 | 106 | 0 | 15 |
| Northern Uzbek | uz | 99 | 95 | 4 | 11 |
| Kazakh | kk | 126 | 126 | 0 | 40 |
| Bashkir | ba | 126 | 125 | 1 | 41 |
| Tatar | tt | 118 | 117 | 1 | 35 |
| Thai | th + Unicode block | 125 | 110 | 15 | 69 |
| Lao | lo + Unicode block | 127 | 76 | 51 | 76 |
| Finnish | fi | 96 | 96 | 0 | 11 |
| Estonian | et | 92 | 92 | 0 | 13 |
| Hungarian | hu | 141 | 120 | 21 | 17 |
| Vietnamese | vi | 220 | 220 | 0 | 120 |
| Khmer | km + Unicode block | 145 | 64 | 81 | 74 |
| Japanese | ja, Han characters from JIS X 0208 | 6627 | 4311 | 2316 | 6501 |
| Korean | KS X 1001 + CLDR punctuation and digits | 2443 | 2165 | 278 | 2154 |
| Georgian | ka + Unicode block | 169 | 84 | 85 | 130 |
| Basque | eu | 98 | 98 | 0 | 12 |
| Haitian | ht | 101 | 101 | 0 | 12 |
| Papiamento | pap | 99 | 99 | 0 | 16 |
| Kabuverdianu | kea | 98 | 97 | 1 | 12 |
| Tok Pisin | tpi | 67 | 67 | 0 | 10 |
| Swahili | sw | 74 | 74 | 0 | 10 |

Tier 1 tokens summed over languages: 37,299, distinct: 18,372


### Check: do byte fragments drop to 0% on spoken Korean?

Spoken sentences are cut with the unchanged Qwen3 tokenizer, and each byte-fragment token (a token that is not a character on
its own) is traced to the character it came from. Fragments are split by position: the first character of a word (after a
space) or a character inside a word. Any fragment tier 1 does not cover is examined before deciding.

In [6]:
covered = set(qwen_surface) | set(tier1["Korean"])
fragment_tokens = total_tokens = 0
causes = collections.Counter()
for text in korean_texts[:20000]:
    encoding = qwen(text, add_special_tokens=False, return_offsets_mapping=True)
    total_tokens += len(encoding["input_ids"])
    for token_id, (start, end) in zip(encoding["input_ids"], encoding["offset_mapping"]):
        if "�" not in qwen.decode([token_id]):
            continue
        fragment_tokens += 1
        position = start
        while position < len(text) and text[position] == " ":
            position += 1
        if position < len(text):
            causes[(text[position], position > 0 and text[position - 1] == " ")] += 1
print(f"Qwen3 as is: byte-fragment tokens {fragment_tokens / total_tokens:.1%} ({fragment_tokens:,}/{total_tokens:,}), 20,000 sentences")
rows = []
for label, word_start in (("first character of a word", True), ("character inside a word", False)):
    forms = {ch: count for (ch, start), count in causes.items() if start == word_start}
    in_qwen = sum(count for ch, count in forms.items() if ch in qwen_surface)
    in_tier1 = sum(count for ch, count in forms.items() if ch not in qwen_surface and ch in covered)
    rest = sum(count for ch, count in forms.items() if ch not in covered)
    rows.append([label, sum(forms.values()), in_qwen, in_tier1, rest])
table(rows, ["Fragment position", "Fragments", "Character is a single Qwen3 token", "Character is in tier 1", "Not covered"])

Qwen3 as is: byte-fragment tokens 14.5% (84,206/580,161), 20,000 sentences


| Fragment position | Fragments | Character is a single Qwen3 token | Character is in tier 1 | Not covered |
|---|---|---|---|---|
| first character of a word | 80663 | 80626 | 4 | 33 |
| character inside a word | 3543 | 3358 | 14 | 171 |